# 第07章《信任的边界》C2组（G18）人工复核 Notebook v2.0

本 Notebook 在收到 A1-G1 正文 Notebook、B1-G19 验证 Notebook/报告、B2-G20 验证报告后更新。

**范围说明**
- A1 上传 Notebook：3 个单元，1 个 Markdown 单元、2 个空代码单元；上传文件 Git blob SHA-1 与 B1/B2 报告所列 A1 blob 一致。
- B1 Notebook：本轮已读取 13 个执行过的代码单元和 T01–T10 输出；本 Notebook 独立重算关键反例与数值。
- B2 Notebook 未上传；A2 正文/完整实验 Notebook 未上传。B1 T06 嵌入 A2 原 7.4 单元 39，足以重算该单元，但不替代全章复现。
- 7.3 本阶段可暂空；本 Notebook 只复核已写的 A1 PDE 例子。

以下代码均自包含，不读取外部文件。

## 1. A1 文件结构与版本核对记录

A1 实际结构：3 cells = 1 markdown + 2 empty code cells；无执行计数/输出。上传文件 blob SHA-1：
`31697239cb36e6beaab40b6e2172fe2705d53aec`。

该值与 B1/B2 报告记录的 A1 blob 相同。此单元仅记录已核对的元数据，不会假装在此重新下载 GitHub 文件。

In [1]:
a1_cell_count = 3
a1_markdown_cells = 1
a1_code_cells = 2
a1_nonempty_code_cells = 0
a1_git_blob_sha1 = "31697239cb36e6beaab40b6e2172fe2705d53aec"

assert a1_cell_count == a1_markdown_cells + a1_code_cells
assert a1_nonempty_code_cells == 0
print({
    "cells": a1_cell_count,
    "markdown": a1_markdown_cells,
    "code": a1_code_cells,
    "nonempty_code": a1_nonempty_code_cells,
    "A1_blob_sha1": a1_git_blob_sha1
})


{'cells': 3, 'markdown': 1, 'code': 2, 'nonempty_code': 0, 'A1_blob_sha1': '31697239cb36e6beaab40b6e2172fe2705d53aec'}


## 2. A1 7.1：有限类 Hoeffding 界数值

对 \(M=21,\delta=.05\)，计算 \(\epsilon=\sqrt{\ln(2M/\delta)/(2n)}\)。这只核验界宽度，不代表单次预测正确概率。

In [2]:
import math
import numpy as np

M, delta = 21, 0.05
rows = []
for n in (20, 80, 320, 1000):
    eps = math.sqrt(math.log(2*M/delta)/(2*n))
    rows.append((n, eps))
for row in rows:
    print(f"n={row[0]:4d}  epsilon={row[1]:.10f}")
assert np.isclose(rows[0][1]/rows[1][1], 2.0)
assert np.isclose(rows[1][1]/rows[2][1], 2.0)
assert np.isclose(rows[-1][1], 0.05802327934474817)


n=  20  epsilon=0.4102865429
n=  80  epsilon=0.2051432715
n= 320  epsilon=0.1025716357
n=1000  epsilon=0.0580232793


## 3. A1 7.1：VC 维证明不等号方向

从 \(A\le n\epsilon^2/8\) 应解得 \(\epsilon\ge\sqrt{8A/n}\)。B1 T03 的设置下，错误地取一半会让尾界远大于 \(\delta\)。

In [3]:
n, d, delta = 100, 1, 0.05
A = d*math.log(2*math.e*n/d) + math.log(4/delta)
radius = math.sqrt(8*A/n)
wrong_eps = radius/2
wrong_tail = 4*(2*math.e*n/d)**d*math.exp(-n*wrong_eps**2/8)
correct_tail = 4*(2*math.e*n/d)**d*math.exp(-n*radius**2/8)
print(f"correct radius={radius:.10f}")
print(f"half radius={wrong_eps:.10f}")
print(f"tail at half radius={wrong_tail:.6f}")
print(f"tail at correct radius={correct_tail:.6f}")
assert np.isclose(radius, 0.9243524869322057)
assert wrong_tail > delta
assert np.isclose(correct_tail, delta)


correct radius=0.9243524869
half radius=0.4621762435
tail at half radius=150.584702
tail at correct radius=0.050000


## 4. A1 7.2：概念漂移下含 λ 的域适应界

用单输入点、源域标签 0、目标域标签 1、常数分类器 \(h_0,h_1\) 构造反例。输入边缘 divergence=0，但 λ=1；含 λ 的标准界仍成立。

In [4]:
source_risks = np.array([0.0, 1.0])  # h0, h1
target_risks = np.array([1.0, 0.0])
divergence = 0.0
lambda_joint = float(np.min(source_risks + target_risks))
rhs = source_risks + divergence/2 + lambda_joint
print("source risks:", source_risks)
print("target risks:", target_risks)
print("HΔH divergence:", divergence)
print("lambda:", lambda_joint)
print("bound RHS for h0,h1:", rhs)
assert lambda_joint == 1.0
assert np.all(target_risks <= rhs)


source risks: [0. 1.]
target risks: [1. 0.]
HΔH divergence: 0.0
lambda: 1.0
bound RHS for h0,h1: [1. 2.]


## 5. A2 7.1：阈值实验的解析风险

A2/B1 使用 \(a=.53,\eta=.1\)，候选阈值 \(t\) 的风险为 \(R(t)=\eta+(1-2\eta)|t-a|\)。

In [5]:
a, eta = 0.53, 0.1
thresholds = np.arange(21)/20
risks = eta + (1-2*eta)*np.abs(thresholds-a)
print("threshold 0.5 analytic risk:", risks[10])
assert np.isclose(risks[10], 0.124)


threshold 0.5 analytic risk: 0.12400000000000003


## 6. A2 7.2：固定网格正弦多项式复算

沿用 B1 T04 的 Polynomial.fit 设置，并与 Chebyshev 基交叉比较。

In [6]:
train = np.linspace(-1.0, 1.0, 41)
inside = (train[:-1] + train[1:])/2
outside = np.linspace(2.0, 3.0, 101)
poly_results = []
for degree in (3, 5, 8):
    model = np.polynomial.Polynomial.fit(train, np.sin(train), degree)
    alt = np.polynomial.Chebyshev.fit(train, np.sin(train), degree)
    for label, points in (("train", train), ("ID", inside), ("OOD", outside)):
        pred = model(points)
        rmse = float(np.sqrt(np.mean((pred-np.sin(points))**2)))
        basis_gap = float(np.max(np.abs(pred-alt(points))))
        poly_results.append((degree, label, rmse, basis_gap))
for row in poly_results:
    print(f"degree={row[0]} {row[1]:5s} RMSE={row[2]:.12g} basis_gap={row[3]:.3g}")
assert np.isclose(poly_results[2][2], 0.7273004622081055, atol=1e-10)
assert np.isclose(poly_results[5][2], 0.13652718726905713, atol=1e-10)
assert np.isclose(poly_results[8][2], 0.014932200902102545, atol=1e-9)
assert max(r[3] for r in poly_results) < 1e-8


degree=3 train RMSE=0.000341122360156 basis_gap=6.66e-16
degree=3 ID    RMSE=0.000312070219151 basis_gap=7.77e-16
degree=3 OOD   RMSE=0.727300462208 basis_gap=1.02e-14
degree=5 train RMSE=2.11223254189e-06 basis_gap=2.11e-15
degree=5 ID    RMSE=1.93932769811e-06 basis_gap=1.78e-15
degree=5 OOD   RMSE=0.136527187269 basis_gap=6.05e-13
degree=8 train RMSE=7.44345983606e-09 basis_gap=2.22e-15
degree=8 ID    RMSE=7.05120182598e-09 basis_gap=2.33e-15
degree=8 OOD   RMSE=0.0149322003232 basis_gap=8.27e-10


## 7. A2 7.4：逐字重建原单元 39 的核心逻辑

注意：这不是完整 A2 Notebook；它重建的是 B1 T06 嵌入并逐字执行的 A2 7.4 单元核心算法。

In [7]:
rng = np.random.default_rng(2026)

def generate_data(x_range, n, noise_std=0.3):
    x = rng.uniform(x_range[0], x_range[1], n)
    y = np.sin(2*np.pi*x) + rng.normal(0, noise_std, n)
    return x, y

def model_predict(x):
    return np.sin(2*np.pi*x) + 0.1

def compute_q_A2(x_cal, y_cal, alpha=0.2):
    residuals = np.abs(y_cal-model_predict(x_cal))
    residuals.sort()
    n = len(residuals)
    k = int(np.ceil((n+1)*(1-alpha)))
    k = min(k, n)  # A2 原实现的边界裁剪 bug
    return float(residuals[k-1])

def evaluate(x_test, y_test, q):
    pred = model_predict(x_test)
    lower, upper = pred-q, pred+q
    return float(np.mean((y_test>=lower)&(y_test<=upper))), float(np.mean(upper-lower))

x_cal, y_cal = generate_data([-1,1], 1000)
q = compute_q_A2(x_cal, y_cal, .2)
x_id, y_id = generate_data([-1,1], 1000)
x_ood, y_ood = generate_data([2,3], 1000)
picp_id, mpiw_id = evaluate(x_id, y_id, q)
picp_ood, mpiw_ood = evaluate(x_ood, y_ood, q)
print(f"q={q:.12f}")
print(f"ID PICP={picp_id:.3f}; MPIW={mpiw_id:.12f}")
print(f"OOD PICP={picp_ood:.3f}; MPIW={mpiw_ood:.12f}")
assert np.isclose(q, 0.41067278696708653)
assert np.isclose(picp_id, 0.813)
assert np.isclose(picp_ood, 0.801)
assert np.isclose(mpiw_id, mpiw_ood)


q=0.410672786967
ID PICP=0.813; MPIW=0.821345573934
OOD PICP=0.801; MPIW=0.821345573934


## 8. A2 7.4：为什么本次 OOD 没有覆盖塌陷

由生成式直接得到 \(Y-\hat\mu(X)=\epsilon-.1\)，所以分数 \(S=|\epsilon-.1|\) 与 X 无关。下面计算固定 q 的解析覆盖，并构造真正的残差偏移对照（+1.5）；后者是 C2/B1 的独立机制对照，不是 A2 原实验。

In [8]:
def Phi(z):
    return 0.5*(1.0+math.erf(z/math.sqrt(2.0)))

coverage_same_score = Phi((q+0.1)/0.3)-Phi((-q+0.1)/0.3)
rng_ref = np.random.default_rng(7419)
eps = rng_ref.normal(0.0, 0.3, 100000)
sim_same = float(np.mean(np.abs(eps-0.1)<=q))
sim_shift = float(np.mean(np.abs(eps+1.5-0.1)<=q))
coverage_shift_theory = Phi((q-1.4)/0.3)-Phi((-q-1.4)/0.3)
print(f"fixed-q theoretical coverage, original residual={coverage_same_score:.9f}")
print(f"independent simulation, original residual={sim_same:.6f}")
print(f"B-constructed +1.5 residual shift: theory={coverage_shift_theory:.9f}, simulation={sim_shift:.6f}")
assert np.isclose(coverage_same_score, 0.8054450453191193)
assert abs(sim_same-coverage_same_score)<0.01
assert abs(sim_shift-coverage_shift_theory)<0.01
assert sim_shift < 0.01


fixed-q theoretical coverage, original residual=0.805445045
independent simulation, original residual=0.805700
B-constructed +1.5 residual shift: theory=0.000487301, simulation=0.000410


## 9. A2 `k=min(k,n)` 的小样本边界 bug

n=5、α=.1 时 k=6=n+1。正确 split conformal 阈值为 \(+\infty\)，而不是最大校准分数。

In [9]:
n_cal, alpha = 5, 0.1
scores = np.arange(1.0, n_cal+1)
k = int(np.ceil((n_cal+1)*(1-alpha)))
q_A2 = float(np.sort(scores)[min(k,n_cal)-1])
q_correct = float("inf") if k>n_cal else float(np.sort(scores)[k-1])
coverage_A2 = n_cal/(n_cal+1)  # continuous iid scores, clipped to max calibration score
print(f"k={k}, q_A2={q_A2}, q_correct={q_correct}")
print(f"marginal coverage with clipped max={coverage_A2:.6f}; target={1-alpha:.2f}")
assert k == 6
assert q_A2 == 5.0
assert math.isinf(q_correct)
assert coverage_A2 < 1-alpha


k=6, q_A2=5.0, q_correct=inf
marginal coverage with clipped max=0.833333; target=0.90


## 10. B1 T09：重复校准的边际覆盖

2000 批独立校准集，每批 99 个分数、200 个测试分数；理论边际覆盖 \(k/(n+1)=.9\)。单批覆盖可以低于 .9，不等于定理失败。

In [10]:
rng_mc = np.random.default_rng(7420)
reps, ncal, ntest = 2000, 99, 200
cal = np.abs(rng_mc.normal(-0.1,0.3,(reps,ncal)))
k = int(np.ceil((ncal+1)*0.9))
qs = np.partition(cal,k-1,axis=1)[:,k-1]
test = np.abs(rng_mc.normal(-0.1,0.3,(reps,ntest)))
cover = (test<=qs[:,None]).mean(axis=1)
mean_cover = float(cover.mean())
mc_se = float(cover.std(ddof=1)/np.sqrt(reps))
q025, q975 = np.quantile(cover,[.025,.975])
print(f"k={k}; exact marginal={k/(ncal+1):.6f}")
print(f"mean empirical coverage={mean_cover:.7f}; MC SE={mc_se:.8f}")
print(f"single-batch 2.5%/97.5% quantiles={q025:.3f}/{q975:.3f}")
assert abs(mean_cover-0.9) < 5*mc_se


k=90; exact marginal=0.900000
mean empirical coverage=0.8998975; MC SE=0.00079689
single-batch 2.5%/97.5% quantiles=0.825/0.960


## 11. A1 7.3：参数化 Poisson 解族的 POD 快照秩

解析解对任意 μ 都是两个固定空间函数的线性组合，所以快照矩阵秩不超过 2。

In [11]:
x = np.linspace(0.0,1.0,201)
mus = np.array([.1,.2,.3,.4,.5])
snapshots = (np.sin(np.pi*x[:,None])*mus[None,:]**2/np.pi**2
             + x[:,None]*(1-x[:,None])*mus[None,:])
U, singular_values, Vt = np.linalg.svd(snapshots,full_matrices=False)
u_mu1 = np.sin(np.pi*x)/np.pi**2 + x*(1-x)
proj2 = U[:,:2]@(U[:,:2].T@u_mu1)
rel_error = float(np.linalg.norm(u_mu1-proj2)/np.linalg.norm(u_mu1))
print("singular values:", singular_values)
print(f"mu=1 two-mode projection relative error={rel_error:.3e}")
assert singular_values[2]/singular_values[0] < 1e-13
assert rel_error < 1e-12


singular values: [2.22341159e+00 2.53801651e-03 1.50225515e-16 5.68438335e-17
 2.27237023e-17]
mu=1 two-mode projection relative error=6.145e-16


## 12. B2 T09：二分类标签与概率区间的集合检查

若标签空间为 {0,1}，区间 [.67,.97] 不包含任何可能标签。它不能被解释为对二分类标签的覆盖预测集。

In [12]:
interval = (0.67,0.97)
labels = (0,1)
contained = {y: interval[0] <= y <= interval[1] for y in labels}
print("label membership:", contained)
assert contained == {0:False,1:False}


label membership: {0: False, 1: False}


## 13. 汇总

本 Notebook 已执行：有限类界、VC 方向、域适应 λ 反例、A2 阈值风险、A2 正弦拟合、A2 原 7.4 核心单元、残差分布机制、k=n+1 边界、重复校准、A1 POD 秩、二分类区间对象。

未执行/未取得：B2 Notebook、A2 完整 Notebook、A2 正文原文件、A 组修订版本和用户指定 commit 的完整 tree。

In [13]:
checks = {
    "A1_empty_code_cells": a1_nonempty_code_cells == 0,
    "VC_sign": wrong_tail > delta and np.isclose(correct_tail,delta),
    "domain_adaptation_lambda": lambda_joint == 1.0,
    "A2_T06_output": np.isclose(picp_id,.813) and np.isclose(picp_ood,.801),
    "A2_small_n_quantile_bug": math.isinf(q_correct) and coverage_A2 < .9,
    "A1_POD_rank2": singular_values[2]/singular_values[0] < 1e-13,
    "binary_interval_invalid": not any(contained.values()),
}
print(checks)
assert all(checks.values())
print("C2 G18 independent checks: PASS")


{'A1_empty_code_cells': True, 'VC_sign': np.True_, 'domain_adaptation_lambda': True, 'A2_T06_output': np.True_, 'A2_small_n_quantile_bug': True, 'A1_POD_rank2': np.True_, 'binary_interval_invalid': True}
C2 G18 independent checks: PASS
